In [ ]:
# --- ENVIRONMENT SETUP: Environment-Aware Paths ---
import sys, os
from pathlib import Path

try:
    # Standard Colab setup
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(str(REPO_ROOT))
except ImportError:
    # Local fallback (assumes running from explorations/ or experiments/)
    cur = Path().resolve()
    REPO_ROOT = cur.parent if cur.name in ('explorations', 'experiments') else cur.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config.paths import GOLD, BRONZE, SILVER, DATA_LAKE, MODELS, PRETRAINED, TRAINED, OPS, MLFLOW_TRACKING_URI, REPOS


# 012_yolo_polyp_detection — YOLO Data Ingestion Pipeline

This notebook runs on **CPU**. It converts Silver data to YOLO format in the Gold layer.


In [ ]:
# Cell 1: Colab Environment Setup
from google.colab import drive, runtime
drive.mount("/content/drive")

# Clone code repos from GitHub
!git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning

import os, sys
sys.path.insert(0, "/content/deep-learning")
os.chdir("/content/deep-learning/experiments/012_yolo_polyp_detection")


In [ ]:
import yaml

from transforms.medical.polyp_silver import process_polyp_dataset
from transforms.gold.yolo_gold import prepare_gold_yolo

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

experiment_name = config["experiment"]["name"]
data_cfg = config["data"]
print(f"Experiment: {experiment_name}")


In [ ]:
# 1. Ensure Bronze -> Silver for each source dataset
silver_datasets = []
for source in data_cfg.get("sources", []):
    ds_name = source["name"]
    try:
        process_polyp_dataset(ds_name, "images", "masks")
    except Exception as e:
        print(f"Silver for {ds_name}: {e} (may already exist)")
    silver_datasets.append(ds_name)
print(f"Silver datasets ready: {silver_datasets}")


In [ ]:
# 2. Generate Experiment-Specific Gold Splits (YOLO format)
gold_cfg = data_cfg.get("gold", {})
prepare_gold_yolo(
    experiment_name=experiment_name,
    silver_datasets=silver_datasets,
    resolution=tuple(gold_cfg.get("resolution", [640, 640])),
)
print("✅ Gold data creation complete!")


In [ ]:
# Release Colab runtime
from google.colab import runtime
runtime.unassign()
